In [10]:
import torch
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
import numpy as np
import random
import os
import pandas as pd
import torch.nn as nn
from collections import defaultdict
import seaborn as sns
import copy


# from GraphRicciCurvature.OllivierRicci import OllivierRicci
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.integrate import simps
import pickle
import time
import pandas as pd
import sys
sys.path.append("..")

import tools.utils as utils
from RicciCurvature.OllivierRicci import OllivierRicci
from tools.FC_linear import FC_Linear
from tools.graph_curvature import graph_curvature_main_torch
from tools.draw_net import DrawNN
from tools.edge_remove import Edge_Remove


np.set_printoptions(threshold=np.inf)
torch.set_printoptions(threshold=torch.inf)

import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [17]:
a = torch.tensor([[1,2,0,3]])
w = torch.tensor([[1,2,3,4,5,6,7,8]])
e = torch.arange(0, 8, 2)
print(e)

a[:,]*w[:,e]

tensor([0, 2, 4, 6])


tensor([[ 1,  6,  0, 21]])

In [ ]:
seed = 59
    
# set random seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
data_train = MNIST('./data/mnist',
                  train=True,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))

data_test = MNIST('./data/mnist',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))




layers = [2, 4, 5, 6, 7]

model_zoo = {
    2: [784, 20, 15, 10],
    21: [784, 200, 150, 10],
    4: [784, 15, 25, 20, 15, 10],
    5: [784, 20, 30, 30, 20, 15, 10],
    6: [784, 20, 30, 30, 35, 20, 15, 10],
    7: [784, 30, 30, 40, 50, 30, 25, 20, 10]
}

selected_classes = [0,1,2,3,4,5,6,7,8,9]

In [ ]:
def standard_PGD(model, images, labels, device, eps=11/255, alpha=2/255, iters=40):
    images = images.to(device)
    labels = labels.to(device)
    loss = nn.CrossEntropyLoss()
        
    ori_images = images.data
        
    for i in range(iters) :    
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        cost = loss(outputs, labels).to(device)
        cost.backward()

        adv_images = images + alpha*images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach_()
            
    return images


In [ ]:
def test_clean(n, loader, device = 'cuda'):
    n.eval()
    total_correct = 0
    
    for i, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)
        output = n(images)
        pred = output.detach().max(1)[1]
        total_correct += pred.eq(labels.view_as(pred)).sum()

    # print(f'Test Accuracy for label {l}: {(float(total_correct) / len(loader.dataset)):.3f}')
    
    acc = float(total_correct) / len(loader.dataset)
    return acc



def test_adversarial(net, loader, eps=.1, alpha=.1, iters=100, device = 'cuda'):
    # prepare model for testing (only important for dropout, batch norm, etc.)
    net.eval()
    
    correct = 0

    for data, target in loader:

        data = standard_PGD(net, data, target, device, eps=eps, alpha=alpha, iters=iters)
        data, target = data.to(device), target.to(device)

        output = net(data)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum()
    
    # print('Test set: Avg. Accuracy: {}/{} ({:.2f}%)'.format(
    #     correct, len(loader.dataset),
    #     (100. * correct / len(loader.dataset))))
    
    return correct / len(loader.dataset)

In [ ]:
def test(n, loader, eps, alpha, iters, device):    
    n.eval()
    robust_pair = defaultdict(list)
    succ_pair = defaultdict(list)
 
    
    for l in selected_classes:
        for i, (images, labels) in enumerate(loader[l]):
            images = images.to(device)
            labels = labels.to(device)
            output = n(images)
            pred = output.detach().max(1)[1]
            
            adv_img = standard_PGD(n, images, labels, device, eps, alpha, iters)
            adv_out = n(adv_img)
            adv_pred = adv_out.detach().max(1)[1]
 
            robust_l = pred.eq(labels.view_as(pred)) & adv_pred.eq(labels.view_as(adv_pred))
            succ_l = pred.eq(labels.view_as(pred)) & ~adv_pred.eq(labels.view_as(adv_pred))

            succ_pair[l].append((images[succ_l].cpu(), adv_img[succ_l].cpu()))
            robust_pair[l].append((images[robust_l].cpu(), adv_img[robust_l].cpu()))

        # print(f'Finish label {l}....')

    return succ_pair, robust_pair

def get_fraction(curvature, b, dims):
    c = []
    layer_num = len(dims) - 1
    neg = np.zeros((layer_num), dtype=np.float32)
    top_neg = np.zeros((layer_num), dtype=np.float32)
    total_e = np.zeros((layer_num), dtype=np.float32)
    # neg = 0.
    # total_e = 0.
    
    for batch in range(b):
        ricci_curv = np.array(curvature[batch])
        for (i, j, curr) in ricci_curv:
            if curr > 1:
                continue
    
            l = int(i)
            if curr < 0:
                neg[l] += 1
            if curr < -10:
                top_neg[l] += 1
            total_e[l] += 1
            c.append(curr)
    return neg, total_e, top_neg, c


In [ ]:
# For FC
def layerwise_shortest_path_torch(dims, weights, device='cpu'):
    batch_size, edge_num = weights.shape
    num_layers = len(dims)
    paths = {}

    weight_idx = 0
    for i in range(num_layers - 1):
        src_size, dst_size = dims[i], dims[i+1]
        direct_dist = weights[:, weight_idx:weight_idx+src_size*dst_size]
        direct_dist = direct_dist.reshape(src_size, dst_size)
        # inf = torch.tensor(float('inf'), device=device)
        # paths[(i, i+1)] = np.where(direct_dist > 0, direct_dist, float('inf'))
        paths[(i, i+1)] = direct_dist
        weight_idx += src_size * dst_size

    return paths

In [ ]:
def get_top_c(curvature, b, prefix_dims, threshold = -50):
    c = []
    shallow_neg_e = set()
    deep_neg_e = set()
    pos_e = set()
    
    for batch in range(b):
        ricci_curv = np.array(curvature[batch])
        for (i, j, curr) in ricci_curv:
            c.append((i,j,curr))

    c.sort(key=lambda x: x[2])
    
    for (i,j,curr) in c:
        i_layer = np.searchsorted(prefix_dims, i, side='right') - 1
        i1 = (int)(i)
        j1 = (int)(j)
        if (i_layer == 1 and curr < 0):
            shallow_neg_e.add((i1,j1))
        elif (curr < 0):
            deep_neg_e.add((i1,j1))
        elif (curr > 0):
            pos_e.add((i1,j1))
        
    return c, shallow_neg_e, deep_neg_e, pos_e



def save_shortest_paths_to_excel(shortest_paths, output_dir="shortest_paths_excel"):
    os.makedirs(output_dir, exist_ok=True)

    # Define the Excel file name
    output_file = f"{output_dir}/shortest_paths.xlsx"
    
    # Create an Excel writer
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for (start, end), numpy_array in shortest_paths.items():
            # Loop through each 2D slice (batch) of the 3D NumPy array
            for i in range(numpy_array.shape[0]):
                df = pd.DataFrame(numpy_array[i].astype(np.float32))
                df.columns = [f'Node_{j}' for j in range(df.shape[1])]
                df.index = [f'Node_{j}' for j in range(df.shape[0])]
                # if (start == 1 and end == 2):
                #     print(df)
                #     print('='*150)
                # Save each batch as a separate sheet with a unique name
                sheet_name = f'shortest_path_{start}_to_{end}_Batch_{i+1}'
                df.to_excel(writer, sheet_name=sheet_name, index=True, index_label='Source Node')

            

In [ ]:
train_loader, test_loader, valid_loader, valid_dataset, test_dataset = utils.get_new_data(selected_classes, data_train, data_test, test_bs=5000, valid_num=5000)

sep_dataloader = utils.sep_label(test_dataset, selected_classes, bs=5000)

eps = [0.03, 0.07, 0.1, 0.2]
Q = [1]
# eps = [0.1]

model_type = 'fc'
model_pre_name = 'ori'
metric = 'q_inv'
res_path = 'res/' + metric + '/'
model_path = 'pgd/models/'
dataset = 'mnist'
alpha = 0
sample_num = 1

In [ ]:
model_full_n = model_type.lower() + model_pre_name.lower()
sample_size = sample_num

if not os.path.exists(res_path):
    os.makedirs(res_path)
    
layers = [2]
if 'big' in model_pre_name.lower():
    layers = [2]
    
dims = model_zoo[2]
    
model_name1 = "pgdtrain_2.pth"
# model_name2 = "decay/best_ori_10l_2.pth"
# model_name3 = "pgdtrain_2.pth"

net_H1 = FC_MD(dims, 2)
net_H1.load_state_dict(torch.load(model_path + model_name1))
net_H1 = net_H1.to(device)
net_full1 = copy.deepcopy(net_H1)

prefix_dims = np.cumsum([0] + dims).tolist()

neural_list = []
nodes_num = 0
edges_num = 0
i = 0
for p in net_H1.parameters():
    if i == 0:
        nodes_num += p.shape[1]
    if i%2 == 0:
        nodes_num += p.shape[0]
        edges_num += (p.shape[0] * p.shape[1])
        neural_list.append(p.shape[0])
    i += 1
    

# remove_frac = [1,2,5,10]
remove_frac = [0.05,0.1,0.2,0.5]

# build model
with open("./edge_remove_acc_fc2_adv.txt", "w+") as ff:
    ff.write(f'For model {model_name1}: \n')
    
    test_cleanacc = test_clean(net_full1, test_loader)
    
    ff.write(f'The clean accuracy for original model is {test_cleanacc}\n')
    
    for e in eps:
        test_advacc = test_adversarial(net_full1, test_loader, eps=e, alpha=2/255, iters=40)
        ff.write(f'The adversary accuracy eps = {e} for original model is {test_advacc}\n\n')
        
        print(f'Current eps {e}: ')
        ff.write(f'Current eps {e}: \n')
        
        succ_pair1, robust_pair1 = test(net_full1, sep_dataloader, eps=e, alpha=2/255, iters=40, device=device)
        
        robust_c = defaultdict(list)
        nonrobust_c = defaultdict(list)
        non_fraction = defaultdict(list)
        rob_fraction = defaultdict(list)
            
        for l in selected_classes:
            print(f'Current label {l}: \n')
            ff.write(f'Current label {l}: \n')
            
            # test_cleanacc_l = test_clean(net_H1, sep_dataloader[l])
            # test_advacc_l = test_adversarial(net_full1, sep_dataloader[l], eps=e, alpha=2/255, iters=40)
            # ff.write(f'For label {l}: The clean accuracy for original model is {test_cleanacc_l}, The adversary accuracy eps = {e} for original model is {test_advacc_l}\n')

            print(f'Robust pair')
            ff.write(f'Robust example: \n')
            # robust images
            count = 0
            for (ori_im, adv_im) in robust_pair1[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array1, nodes_ori1, output1, all_node1 = net_H1.NN_info_batch(img.unsqueeze(0))
 
                    weights1 = output1.detach().clone().to(device)                   
                    weights1[edge_array1 == 0] = 0.
                    
                    # img_a = im_a.to(device)
                    # edge_array1_a, nodes_ori1_a, output1_a, all_node1_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    # weights1_a = output1_a.detach().clone().to(device)                   
                    # weights1_a[edge_array1_a == 0] = 0.

                    weights_inv1 = net_H1.normalization_weight_w2(nodes_ori1, weights1, dims)
                    weights_inv1 = weights_inv1.detach()
                    ricci_curvature1, sp_dict1 = graph_curvature_main_torch(dims, weights_inv1, device=device, alpha=alpha)
                    # weights_inv1_a = net_H1.normalization_weight_w2(nodes_ori1_a, weights1_a, dims)
                    # weights_inv1_a = weights_inv1_a.detach()
                    # ricci_curvature1_a, sp_dict1_a = graph_curvature_main_torch(dims, weights_inv1_a, device=device, alpha=alpha)
                    
                    # e1 = nodes_ori1_a - nodes_ori1
                    c1, shallow_neg_e1, deep_neg_e1, pos_e1 = get_top_c(ricci_curvature1, 1, prefix_dims, threshold = -30)
                    reversed_pos_e1 = list(pos_e1)[::-1]
                    
                    ff.write(f'It has {len(shallow_neg_e1)} shallow negative curvature edges, {len(deep_neg_e1)} deep negative curvature edges, {len(pos_e1)} positive curvature egdes .. \n')

                    ff.write(f'For CE model {count}, the lowest curvature edges are {c1[0][0], c1[0][1], c1[0][2]}: \n')
                    for indx, edge in enumerate(c1):
                        ff.write(f'{edge[0], edge[1], edge[2]}\n')
                        if indx >= 15:
                            break
                    
                    ff.write("\n\n")
                    
                    net_H = FC_MD(dims, 2)
                    net_H = net_H.to(device)

                    for index, rem_f in enumerate(remove_frac):
                        ff.write(f'Remove edge fraction {rem_f}: \n')
                        cur_n = "full" + str(rem_f)
                                
                        net_H.load_state_dict(torch.load(model_path + model_name1))
                        edge_r = Edge_Remove(net_H, dims, rem_f, model_path)
                        edge_r.e_remove(shallow_neg_e1, cur_n + "shallow.pth")
                        
                        # test acc
                        net_new = FC_MD(dims, 2)

                        net_new.load_state_dict(torch.load(model_path + cur_n + "shallow.pth"))
                        net_new = net_new.to(device)
                        
                        acc_clean = test_clean(net_new, test_loader)
                        acc_adv = test_adversarial(net_new, test_loader, eps=e, alpha=2/255, iters=40)
                        
                        ff.write(f'Test Accuracy after remove {(int)(len(shallow_neg_e1)*rem_f)} shallow_neg_e1 edges: clean acc {acc_clean}, adv acc {acc_adv:.3f}...\n')
                        
                        net_H.load_state_dict(torch.load(model_path + model_name1))
                        edge_r = Edge_Remove(net_H, dims, rem_f, model_path)
                        edge_r.e_remove(deep_neg_e1, cur_n + "deep.pth")
                        
                        # test acc
                        net_new = FC_MD(dims, 2)

                        net_new.load_state_dict(torch.load(model_path + cur_n + "deep.pth"))
                        net_new = net_new.to(device)
                        
                        acc_clean = test_clean(net_new, test_loader)
                        acc_adv = test_adversarial(net_new, test_loader, eps=e, alpha=2/255, iters=40)
                        
                        ff.write(f'Test Accuracy after remove {(int)(len(deep_neg_e1)*rem_f)} deep_neg_e1 edges: clean acc {acc_clean}, adv acc {acc_adv:.3f}...\n')
                        
                        net_H.load_state_dict(torch.load(model_path + model_name1))
                        edge_r = Edge_Remove(net_H, dims, rem_f, model_path)
                        edge_r.e_remove(reversed_pos_e1, cur_n + "pos.pth")
                        
                        # test acc
                        net_new = FC_MD(dims, 2)

                        net_new.load_state_dict(torch.load(model_path + cur_n + "pos.pth"))
                        net_new = net_new.to(device)
                        
                        acc_clean = test_clean(net_new, test_loader)
                        acc_adv = test_adversarial(net_new, test_loader, eps=e, alpha=2/255, iters=40)
                        
                        ff.write(f'Test Accuracy after remove {(int)(len(reversed_pos_e1)*rem_f)} reversed_pos_e1 edges: clean acc {acc_clean}, adv acc {acc_adv:.3f}...\n')
                            
                        ff.write("\n\n")
                    
                    # print('CE')
                    # save_shortest_paths_to_excel(sp_dict1, "./pandas/CE/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                    
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
            
            # print(f'nonRobust pair')
            # ff.write(f'nonRobust example: \n')
            # count = 0
            # for (ori_im, adv_im) in succ_pair1[l]:
            #     if (count >= sample_size):
            #         break
            #     for (im, im_a) in zip(ori_im, adv_im):
            #         img = im.to(device)
            #         edge_array1, nodes_ori1, output1, all_node1 = net_H1.NN_info_batch(img.unsqueeze(0))
 
            #         weights1 = output1.detach().clone().to(device)                   
            #         weights1[edge_array1 == 0] = 0.
                    
            #         # img_a = im_a.to(device)
            #         # edge_array1_a, nodes_ori1_a, output1_a, all_node1_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
            #         # weights1_a = output1_a.detach().clone().to(device)                   
            #         # weights1_a[edge_array1_a == 0] = 0.

            #         weights_inv1 = net_H1.normalization_weight_w2(nodes_ori1, weights1, dims)
            #         weights_inv1 = weights_inv1.detach()
            #         # weights_inv1_a = net_H1.normalization_weight_w2(nodes_ori1_a, weights1_a, dims)
            #         # weights_inv1_a = weights_inv1_a.detach()
                    
            #         # e1_non = nodes_ori1_a - nodes_ori1
                    
            #         ricci_curvature1, sp_dict1 = graph_curvature_main_torch(dims, weights_inv1, device=device, alpha=alpha)
                    
            #         c1, shallow_neg_e1, deep_neg_e1, pos_e1 = get_top_c(ricci_curvature1, 1, prefix_dims, threshold = -30)
            #         reversed_pos_e1 = list(pos_e1)[::-1]
                    
            #         ff.write(f'It has {len(shallow_neg_e1)} shallow negative curvature edges, {len(deep_neg_e1)} deep negative curvature edges, {len(pos_e1)} positive curvature egdes .. \n')

            #         ff.write(f'For CE model {count}, the lowest curvature edges are {c1[0][0], c1[0][1], c1[0][2]}: \n')
            #         for indx, edge in enumerate(c1):
            #             ff.write(f'{edge[0], edge[1], edge[2]}\n')
            #             if indx >= 15:
            #                 break
                    
            #         ff.write("\n\n") 
                    
            #         for index, rem_f in enumerate(remove_frac):
            #             ff.write(f'Remove edge fraction {rem_f}: \n')
            #             cur_n = "full" + str(rem_f)
            #             model_n = cur_n + ".pth"
                                
            #             net_H1.load_state_dict(torch.load(model_path + model_name1))
            #             edge_r = Edge_Remove(net_H1, dims, rem_f, model_path)
            #             edge_r.e_remove(shallow_neg_e1, cur_n + "shallow.pth")
                        
            #             # test acc
            #             net_new = FC_MD(dims, 2)

            #             net_new.load_state_dict(torch.load(model_path + cur_n + "shallow.pth"))
            #             net_new = net_new.to(device)
                        
            #             acc_clean = test_clean(net_new, test_loader)
            #             acc_adv = test_adversarial(net_new, test_loader)
                        
            #             ff.write(f'Test Accuracy after remove {(int)(len(shallow_neg_e1)*rem_f)} shallow_neg_e1 edges: clean acc {acc_clean}, adv acc {acc_adv:.3f}...\n')
                        
            #             net_H1.load_state_dict(torch.load(model_path + model_name1))
            #             edge_r = Edge_Remove(net_H1, dims, rem_f, model_path)
            #             edge_r.e_remove(deep_neg_e1, cur_n + "deep.pth")
                        
            #             # test acc
            #             net_new = FC_MD(dims, 2)

            #             net_new.load_state_dict(torch.load(model_path + cur_n + "deep.pth"))
            #             net_new = net_new.to(device)
                        
            #             acc_clean = test_clean(net_new, test_loader)
            #             acc_adv = test_adversarial(net_new, test_loader)
                        
            #             ff.write(f'Test Accuracy after remove {(int)(len(deep_neg_e1)*rem_f)} deep_neg_e1 edges: clean acc {acc_clean}, adv acc {acc_adv:.3f}...\n')
                        
            #             net_H1.load_state_dict(torch.load(model_path + model_name1))
            #             edge_r = Edge_Remove(net_H1, dims, rem_f, model_path)
            #             edge_r.e_remove(reversed_pos_e1, cur_n + "pos.pth")
                        
            #             # test acc
            #             net_new = FC_MD(dims, 2)

            #             net_new.load_state_dict(torch.load(model_path + cur_n + "pos.pth"))
            #             net_new = net_new.to(device)
                        
            #             acc_clean = test_clean(net_new, test_loader)
            #             acc_adv = test_adversarial(net_new, test_loader)
                        
            #             ff.write(f'Test Accuracy after remove {(int)(len(reversed_pos_e1)*rem_f)} reversed_pos_e1 edges: clean acc {acc_clean}, adv acc {acc_adv:.3f}...\n')
                    
            #             ff.write("\n\n")
                    
            #         count += 1
            #         if (count % 10 == 0):
            #             print(f'Finish {count} graphs....')
                        
            #         if (count >= sample_size):
            #             break
                    

